In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def test_instruct_model(model_id, prompt):
    print(f"========== Testing {model_id} ==========")

    # 1. Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # 2. Load Model (Using bfloat16 to save VRAM)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    # 3. Build the Chat Format
    # This matches the assignment requirement to use the instruction format
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    # 4. Apply Chat Template
    text_with_template = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    print(f"Formatted Prompt:\n{text_with_template}\n")

    # 5. Tokenize and Generate
    model_inputs = tokenizer([text_with_template], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=150,
        do_sample=False # Deterministic decoding for testing
    )

    # 6. Extract only the newly generated tokens
    generated_ids_only = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids_only, skip_special_tokens=True)[0]

    print(f"Model Answer:\n{response}\n")

    # 7. Memory Cleanup (Crucial if running on a single GPU)
    del model
    del tokenizer
    torch.cuda.empty_cache()

# --- Run the tests ---
# You can change this prompt to any of your 10 English queries
test_prompt = "Explain why leaves are green in two sentences."

# Note: You may need to run `huggingface-cli login` in your terminal
# if mistral requires accepting their terms on the HF Hub.
models_to_test = [
    "Qwen/Qwen2.5-7B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3"
]

for model_name in models_to_test:
    test_instruct_model(model_name, test_prompt)

In [ ]:
import json
import re
import string
from transformers import AutoTokenizer

# ============================================================
# REGEXES
# ============================================================

# Hebrew unicode block
HEBREW_RE = re.compile(r'[\u0590-\u05FF]')

# Scripts we want to reject
LATIN_RE = re.compile(r'[a-zA-Z]')
CYRILLIC_RE = re.compile(r'[\u0400-\u04FF]')
CJK_RE = re.compile(r'[\u4E00-\u9FFF]')

# punctuation + digits + whitespace
PUNCT = re.escape(string.punctuation)

COMMON_CHARS_RE = re.compile(
    rf'^[\d\s{PUNCT}]+$'
)

# tokenizer artifacts that should NOT disqualify a token
TOKENIZER_ARTIFACTS_RE = re.compile(r'[▁ĠĊ]')

# ============================================================
# TOKEN FILTERING FUNCTION
# ============================================================

def is_allowed_token(decoded_token: str) -> bool:

    # --------------------------------------------------------
    # Remove tokenizer artifacts
    # --------------------------------------------------------

    cleaned = TOKENIZER_ARTIFACTS_RE.sub('', decoded_token)

    # --------------------------------------------------------
    # Empty / whitespace token
    # --------------------------------------------------------

    if cleaned.strip() == "":
        return True

    # --------------------------------------------------------
    # Pure punctuation / numbers / spaces
    # --------------------------------------------------------

    if COMMON_CHARS_RE.fullmatch(cleaned):
        return True

    # --------------------------------------------------------
    # Count scripts
    # --------------------------------------------------------

    hebrew_count = len(HEBREW_RE.findall(cleaned))
    latin_count = len(LATIN_RE.findall(cleaned))
    cyrillic_count = len(CYRILLIC_RE.findall(cleaned))
    cjk_count = len(CJK_RE.findall(cleaned))

    # --------------------------------------------------------
    # Must contain some Hebrew
    # --------------------------------------------------------

    if hebrew_count == 0:
        return False

    # --------------------------------------------------------
    # Reject clearly non-Hebrew tokens
    # BUT allow mixed tokens if Hebrew dominates
    # --------------------------------------------------------

    if latin_count > hebrew_count:
        return False

    if cyrillic_count > 0:
        return False

    if cjk_count > 0:
        return False

    return True


# ============================================================
# MAIN FUNCTION
# ============================================================

def extract_whitelist_hebrew_tokens(model_id, output_filename):

    print(f"\n========== Processing {model_id} ==========")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    vocab = tokenizer.get_vocab()

    allowed_token_ids = []

    for token, token_id in vocab.items():

        try:
            decoded_token = tokenizer.decode(
                [token_id],
                clean_up_tokenization_spaces=False
            )
        except:
            continue

        # ----------------------------------------------------
        # Always allow special tokens
        # ----------------------------------------------------

        if token_id in tokenizer.all_special_ids:
            allowed_token_ids.append(token_id)
            continue

        # ----------------------------------------------------
        # Apply filtering logic
        # ----------------------------------------------------

        if is_allowed_token(decoded_token):
            allowed_token_ids.append(token_id)

    # --------------------------------------------------------
    # Always allow EOS
    # --------------------------------------------------------

    if tokenizer.eos_token_id is not None:
        allowed_token_ids.append(tokenizer.eos_token_id)

    # remove duplicates + sort
    allowed_token_ids = sorted(list(set(allowed_token_ids)))

    # --------------------------------------------------------
    # Save JSON
    # --------------------------------------------------------

    output_data = {
        "model_id": model_id,
        "allowed_token_ids": allowed_token_ids
    }

    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)

    print(f"Whitelist completed!")
    print(f"Saved {len(allowed_token_ids)} tokens.")
    print(f"Output file: {output_filename}")


# ============================================================
# RUN
# ============================================================

extract_whitelist_hebrew_tokens(
    "Qwen/Qwen2.5-7B-Instruct",
    "hebrew_allowed_tokens_qwen.json"
)

extract_whitelist_hebrew_tokens(
    "mistralai/Mistral-7B-Instruct-v0.3",
    "hebrew_allowed_tokens_mistral.json"
)

In [ ]:
import torch
import json

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList
)

# ============================================================
# LOGITS PROCESSOR
# ============================================================

class WhitelistLogitsProcessor(LogitsProcessor):

    def __init__(self, allowed_token_ids, vocab_size):

        self.mask = torch.zeros(vocab_size, dtype=torch.bool)

        self.mask[allowed_token_ids] = True

    def __call__(self, input_ids, scores):

        mask = self.mask.to(scores.device)

        # block forbidden tokens
        scores[:, ~mask] = -float("inf")

        return scores


# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_response(
    model,
    tokenizer,
    prompt,
    logits_processor_list=None
):

    messages = [
        {
            "role": "system",
            "content": "ענה בעברית בלבד."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # IMPORTANT:
    # use chat template correctly
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    input_length = model_inputs.input_ids.shape[1]

    outputs = model.generate(
        **model_inputs,

        max_new_tokens=80,

        do_sample=True,
        temperature=0.7,
        top_p=0.9,

        repetition_penalty=1.1,

        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,

        logits_processor=logits_processor_list
    )

    generated = outputs[0][input_length:]

    decoded = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return decoded.strip()


# ============================================================
# MAIN EVALUATION
# ============================================================

def run_evaluation():

    models_to_test = [
        {
            "id": "Qwen/Qwen2.5-7B-Instruct",
            "json_file": "hebrew_allowed_tokens_qwen.json"
        },
        {
            "id": "mistralai/Mistral-7B-Instruct-v0.3",
            "json_file": "hebrew_allowed_tokens_mistral.json"
        }
    ]

    prompts = [
        "Explain why the sky looks blue during the day.",
        "Give two advantages and two disadvantages of public transportation.",
        "Write a short email asking a professor for an extension on an assignment.",
        "Describe how to make a simple omelette.",
        "What is the difference between supervised and unsupervised learning?",
        "Summarize the story of Cinderella in three sentences.",
        "Suggest three ways to reduce smartphone distraction while studying.",
        "Explain what happens when water boils.",
        "Give a polite refusal to an invitation to a party.",
        "Turn the idea 'practice makes progress' into advice for a student."
    ]

    output_file = "decoding_outputs.jsonl"

    with open(output_file, 'w', encoding='utf-8') as f_out:

        for model_info in models_to_test:

            model_id = model_info["id"]

            print(f"\n========== Loading {model_id} ==========")

            tokenizer = AutoTokenizer.from_pretrained(model_id)

            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=torch.bfloat16,
                device_map="auto"
            )

            model.eval()

            # ------------------------------------------------
            # Load allowed token ids
            # ------------------------------------------------

            with open(model_info["json_file"], 'r', encoding='utf-8') as f:

                allowed_data = json.load(f)

                allowed_token_ids = allowed_data["allowed_token_ids"]

            # ------------------------------------------------
            # Build constrained decoder
            # ------------------------------------------------

            whitelist_processor = WhitelistLogitsProcessor(
                allowed_token_ids,
                model.config.vocab_size
            )

            logits_processor_list = LogitsProcessorList([
                whitelist_processor
            ])

            # ------------------------------------------------
            # Run prompts
            # ------------------------------------------------

            for prompt in prompts:

                print(f"\nPrompt: {prompt}")

                # --------------------------------------------
                # Unconstrained
                # --------------------------------------------

                unconstrained_output = generate_response(
                    model,
                    tokenizer,
                    prompt,
                    logits_processor_list=None
                )

                # --------------------------------------------
                # Constrained
                # --------------------------------------------

                constrained_output = generate_response(
                    model,
                    tokenizer,
                    prompt,
                    logits_processor_list=logits_processor_list
                )

                print("\n--- Unconstrained ---")
                print(unconstrained_output[:200])

                print("\n--- Constrained ---")
                print(constrained_output[:200])

                # --------------------------------------------
                # Save JSONL
                # --------------------------------------------

                result_obj = {
                    "prompt": prompt,
                    "model": model_id,
                    "unconstrained_output": unconstrained_output,
                    "constrained_output": constrained_output
                }

                f_out.write(
                    json.dumps(result_obj, ensure_ascii=False)
                    + '\n'
                )

            # ------------------------------------------------
            # Cleanup GPU memory
            # ------------------------------------------------

            del model
            del tokenizer

            torch.cuda.empty_cache()

    print(f"\nDone!")
    print(f"Results saved to: {output_file}")


# ============================================================
# RUN
# ============================================================

run_evaluation()